# Bekami — expérience QLoRA (piste B)
Question posée : un adaptateur LoRA entraîné sur des paires *signature + docstring → corps* fait-il monter, baisser ou laisser inchangé le score HumanEval de DeepSeek-Coder 1.3B ?

**Attention.** Le jeu fourni (bibliothèque standard Python, ~1 100 exemples) est petit : on attend peu de changement. Cette expérience valide la chaîne complète et détecte une régression. HumanEval a 164 problèmes : un écart de quelques points est dans le bruit.

**Sécurité.** `evaluate.py` exécute du code généré par le modèle. Utilisez une session Colab jetable, sans Drive monté pendant l'évaluation.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Activez un GPU : Exécution > Modifier le type d\'exécution'
print(torch.cuda.get_device_name(0))

## 1. Projet
Sur votre Mac, depuis le dossier du projet : `zip -r ../bekami_llm_starter.zip . -x '*/__pycache__/*' '.git/*' 'data/raw/*' 'checkpoints/*'`, puis téléversez l'archive dans `/content`.

In [ ]:
!mkdir -p /content/bekami_llm && unzip -o -q /content/bekami_llm_starter.zip -d /content/bekami_llm
%cd /content/bekami_llm
!pip install -q -r requirements-ft.txt

## 2. Données : HumanEval (évaluation + décontamination) et jeu d'entraînement

In [ ]:
!python fetch_humaneval.py
!python evaluate.py code --canonical --problems data/eval/humaneval.jsonl   # le banc doit afficher 164/164
!python build_sft.py python-exp --stdlib --max-examples 5000

## 3. Mesure de référence (modèle de base, avant entraînement)

In [ ]:
!python evaluate.py code --hf deepseek-ai/deepseek-coder-1.3b-base --problems data/eval/humaneval.jsonl --out base.jsonl

## 4. Entraînement de l'adaptateur

In [ ]:
!python finetune.py python-exp

## 5. Mesure avec l'adaptateur

In [ ]:
!python evaluate.py code --hf deepseek-ai/deepseek-coder-1.3b-base --adapter python-exp --problems data/eval/humaneval.jsonl --out tuned.jsonl

## 6. Comparaison

In [ ]:
import json
load = lambda p: {r['task_id']: r['passed'] for r in map(json.loads, open(p))}
base, tuned = load('base.jsonl'), load('tuned.jsonl')
gained = sorted(t for t in base if tuned[t] and not base[t])
lost = sorted(t for t in base if base[t] and not tuned[t])
n = len(base)
print(f'base  : {sum(base.values())}/{n} = {sum(base.values())/n:.1%}')
print(f'tuned : {sum(tuned.values())}/{n} = {sum(tuned.values())/n:.1%}')
print('gagnés :', gained)
print('perdus :', lost)
print('Écart-type approximatif d\'un score sur 164 problèmes : ±3 à 4 points. En dessous, ne rien conclure.')

## 7. Sauvegarde de l'adaptateur (optionnel)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r adapters/python-exp/weights /content/drive/MyDrive/bekami_python-exp_weights